<!--
Copyright (c) 2026 Milin Patel
Hochschule Kempten - University of Applied Sciences
Campustage 2026 - Automatisiertes Fahren! Sicher durch Zufall?
MIT License - see LICENSE.
-->


# Der KI-Blick — Wie sieht ein selbstfahrendes Auto?
### Campustage 2026 · *Automatisiertes Fahren! Sicher durch Zufall?*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/campustage-2026-sicher-durch-zufall/blob/main/01_Perception_Systems/notebooks/00_workshop_der_ki_blick.ipynb)

**Milin Patel · Hochschule Kempten, IFM · MIT Licence**

---

Ein selbstfahrendes Auto erkennt seine Umgebung mit **Kamera** und **KI**.
Hier schaust du der KI beim Sehen zu — und findest heraus, **wo sie versagt.**

Bei jeder Zelle links auf **▶** klicken (oder *Umschalt + Enter*). Kein Code lesen — alles läuft über Regler und Knöpfe.


---
## Schritt 0 · Vorbereiten
Einmal ausführen. Lädt die KI (ca. 1 Minute).


In [ ]:
%pip install ultralytics -q
import io, numpy as np, requests
from PIL import Image, ImageDraw, ImageFilter
from ultralytics import YOLO
import ipywidgets as widgets
import matplotlib.pyplot as plt
try:
    from google.colab import output as _co; _co.enable_custom_widget_manager()
except Exception: pass

ORANGE=(232,133,12); ROT=(200,60,60)
model = YOLO("yolov8n.pt")
NAMES = model.names
DE = {"person":"Mensch","car":"Auto","bus":"Bus","truck":"LKW","bicycle":"Fahrrad",
      "motorcycle":"Motorrad","stop sign":"Stoppschild","traffic light":"Ampel","train":"Zug"}
def de(n): return DE.get(n,n)
print("Bereit.")


---
## Schritt 1 · Das Bild

Wir arbeiten das ganze Notebook über mit **einem** Bild.
Standard ist eine Straßenszene. **Optional:** du kannst dein eigenes Foto hochladen
(eine Straße mit Menschen/Autos) — dann nutzt jeder Schritt danach *dein* Bild.

Führe die Zelle aus. Zum Hochladen den Knopf benutzen; sonst einfach das Standardbild lassen.


In [ ]:
def lade_standard():
    r=requests.get("https://ultralytics.com/images/bus.jpg", timeout=20)
    return Image.open(io.BytesIO(r.content)).convert("RGB")

BILD = lade_standard()
knopf = widgets.FileUpload(accept="image/*", multiple=False, description="Eigenes Bild")
def _on(change):
    global BILD
    if knopf.value:
        data=list(knopf.value.values())[0]["content"] if isinstance(knopf.value,dict) else knopf.value[0]["content"]
        BILD=Image.open(io.BytesIO(data)).convert("RGB")
        _zeige(BILD, "Dein Bild ist geladen — es gilt ab jetzt für alle Schritte.")
knopf.observe(_on, names="value")

def _zeige(img, titel):
    plt.close("all"); plt.figure(figsize=(9,6)); plt.imshow(img); plt.axis("off")
    plt.title(titel); plt.show()

display(knopf)
_zeige(BILD, "Das ist unser Bild. (Optional oben ein eigenes hochladen.)")


---
## Schritt 2 · Du zählst zuerst

Schau das Bild oben genau an und **zähl mit den Augen:**

> **Wie viele Menschen siehst du?**

Merk dir die Zahl. Gleich vergleichen wir sie mit der KI.


---
## Schritt 3 · Jetzt schaut die KI

Ein Klick — die KI sucht. Jede **Box** ist ein Fund, mit einer **Konfidenz** (wie sicher sie ist).
Unten steht, **wie viele Menschen** sie erkannt hat. Gleich viele wie du?


In [ ]:
def erkenne(img, schwelle):
    res = model.predict(img, conf=schwelle, verbose=False)[0]
    out = img.copy(); d = ImageDraw.Draw(out); personen=0
    for b in res.boxes:
        x1,y1,x2,y2 = [int(v) for v in b.xyxy[0]]
        cf=float(b.conf[0]); name=de(NAMES.get(int(b.cls[0]),""))
        col = ORANGE if cf>=0.5 else ROT
        d.rectangle([x1,y1,x2,y2], outline=col, width=3)
        d.text((x1+4, max(0,y1-12)), f"{name} {cf:.0%}", fill=col)
        if NAMES.get(int(b.cls[0]))=="person": personen+=1
    return out, personen

bild, n = erkenne(BILD, 0.25)
plt.figure(figsize=(9,6)); plt.imshow(bild); plt.axis("off")
plt.title(f"Die KI erkennt {n} Menschen"); plt.show()


---
## Schritt 4 · Die Grenze entscheidet

Die KI ist sich **nie 100 % sicher** — sie gibt jedem Fund nur eine Zahl.
Ein Mensch legt fest, ab welcher Zahl etwas als *erkannt* gilt: die **Schwelle.**

Links eine **niedrige**, rechts eine **hohe** Schwelle — dasselbe Bild.
Achte darauf, wie rechts **echte Menschen verschwinden.**


In [ ]:
def vergleich_schwelle(niedrig=0.25, hoch=0.70):
    plt.close("all")
    fig,ax=plt.subplots(1,2,figsize=(14,6))
    for a,s in zip(ax,(niedrig,hoch)):
        img,n=erkenne(BILD,s)
        a.imshow(img); a.axis("off")
        a.set_title(f"Schwelle {s:.2f}  →  {n} Menschen erkannt")
    plt.tight_layout(); plt.show()

vergleich_schwelle(0.25, 0.70)


**Jetzt du:** zieh den Regler und beobachte, wie die Zahl der erkannten Menschen kippt.
Zu hoch — echte Menschen fehlen. Zu niedrig — die KI meldet Dinge, die nicht da sind.
**Es gibt keine perfekte Einstellung. Und ein Mensch wählt sie.**


In [ ]:
def regler(Schwelle=0.25):
    plt.close("all"); img,n=erkenne(BILD,Schwelle)
    plt.figure(figsize=(9,6)); plt.imshow(img); plt.axis("off")
    plt.title(f"Schwelle {Schwelle:.2f}  →  {n} Menschen erkannt"); plt.show()
widgets.interact(regler, Schwelle=widgets.FloatSlider(min=0.05,max=0.95,step=0.05,value=0.25,continuous_update=False));


---
## Schritt 5 · Die Welt ist nicht sauber

Auf der Straße ist es nicht immer klar. Hier dasselbe Bild bei **Klar, Nebel und Regen** —
gleiche Schwelle. Sieh, wie die Zahl der erkannten Menschen **einbricht.**


In [ ]:
def mach_wetter(img, art):
    if art=="Nebel":
        n=Image.new("RGB",img.size,(205,205,210))
        return Image.blend(img,n,0.55).filter(ImageFilter.GaussianBlur(2))
    if art=="Regen":
        o=img.copy(); d=ImageDraw.Draw(o); rng=np.random.default_rng(0)
        for _ in range(500):
            x=int(rng.integers(0,img.width)); y=int(rng.integers(0,img.height))
            d.line([(x,y),(x-2,y+12)], fill=(185,190,205))
        return Image.eval(o, lambda p:int(p*0.8))
    return img

def vergleich_wetter(Schwelle=0.25):
    plt.close("all")
    arten=["Klar","Nebel","Regen"]
    fig,ax=plt.subplots(1,3,figsize=(16,5))
    for a,art in zip(ax,arten):
        img,n=erkenne(mach_wetter(BILD,art),Schwelle)
        a.imshow(img); a.axis("off"); a.set_title(f"{art}  →  {n} Menschen")
    plt.tight_layout(); plt.show()

widgets.interact(vergleich_wetter, Schwelle=widgets.FloatSlider(min=0.05,max=0.95,step=0.05,value=0.25,continuous_update=False));


---
## Das nimmst du mit

> **Die KI sieht nicht, was du siehst. Sie sieht Zahlen — und Zahlen kann man täuschen.**

- Sie erkennt Dinge nur mit einer **Wahrscheinlichkeit**, nie sicher.
- Die **Schwelle** legt ein **Mensch** fest — zu hoch, Menschen verschwinden; zu niedrig, Fehlalarme.
- **Nebel und Regen** drücken die Konfidenz unter die Schwelle — die Person ist für die KI weg.

Das Problem ist **nicht gelöst.** Daran wird gearbeitet — vielleicht bald von dir.

---
*Mehr aus diesem Feld: die übrigen Notebooks in diesem Repository (Perception, Failure Analysis, Functional Safety, SOTIF, Cybersecurity, AI Safety) — für alle, die tiefer schauen wollen.*
